<a href="https://colab.research.google.com/github/aligreo/TriEncoder-Unet-Project/blob/main/unet_reem_abdallah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!uv pip install SimpleITK monai

In [ ]:
import os
import random
import warnings
import torch

from msseg_utils import (
    unzip_if_needed,
    collect_dataset,
    stratified_split,
    binarize_label
)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Load Datasets

In [ ]:
import kagglehub
path = kagglehub.dataset_download("hongnguyntriu/mslessegdataset")

MSSEG_TRAIN_ZIP = "/content/drive/MyDrive/MSSEG-Training.zip"
MSSEG_EXTRACT_DIR = "/content/MSSEG-Training"
MSLESSEG_ROOT = path

SEED = 42
random.seed(SEED)

msseg_root = unzip_if_needed(MSSEG_TRAIN_ZIP, MSSEG_EXTRACT_DIR)
msseg_files = collect_dataset(msseg_root, "MSSEG")
mslesseg_files = collect_dataset(MSLESSEG_ROOT, "MSLesSeg")

data_dicts = msseg_files + mslesseg_files
random.shuffle(data_dicts)

print(f"Total combined cases: {len(data_dicts)}")

### Preprocessing and Augmentation Pipeline

In [ ]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    NormalizeIntensityd,
    ConcatItemsd,
    DeleteItemsd,
    RandCropByPosNegLabeld,
    EnsureTyped,
    CropForegroundd,
    RandFlipd,
    RandRotate90d,
    RandScaleIntensityd,
    RandShiftIntensityd,
    RandGaussianNoised,
    RandBiasFieldd,
    Lambdad,
)
from monai.data import PersistentDataset, DataLoader

base_transforms = [
    LoadImaged(keys=["flair", "t1", "t2", "label"]),
    EnsureChannelFirstd(keys=["flair", "t1", "t2", "label"]),
    Orientationd(keys=["flair", "t1", "t2", "label"], axcodes="RAS", labels=None),
    Spacingd(
        keys=["flair", "t1", "t2", "label"],
        pixdim=(1.0, 1.0, 1.0),
        mode=("bilinear", "bilinear", "bilinear", "nearest"),
        padding_mode="zeros",
    ),
    Lambdad(keys="label", func=binarize_label),
    CropForegroundd(keys=["flair", "t1", "t2", "label"], source_key="flair"),
    NormalizeIntensityd(keys=["flair", "t1", "t2"], nonzero=True, channel_wise=True),
    ConcatItemsd(keys=["flair", "t1", "t2"], name="image", dim=0),
    DeleteItemsd(keys=["flair", "t1", "t2"]),
]

train_transforms = Compose(base_transforms + [
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(96, 96, 96),
        pos=3,
        neg=1,
        num_samples=4,
        image_key="image",
        image_threshold=0,
    ),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    RandRotate90d(keys=["image", "label"], prob=0.25, max_k=3),
    RandScaleIntensityd(keys="image", factors=0.15, prob=0.35),
    RandShiftIntensityd(keys="image", offsets=0.10, prob=0.35),
    RandGaussianNoised(keys="image", prob=0.15, mean=0.0, std=0.01),
    RandBiasFieldd(keys="image", prob=0.15, coeff_range=(0.0, 0.03)),
    EnsureTyped(keys=["image", "label"]),
])

val_transforms = Compose(base_transforms + [
    EnsureTyped(keys=["image", "label"]),
])

train_files, val_files = stratified_split(data_dicts, val_fraction=0.2, seed=SEED)

train_ds = PersistentDataset(data=train_files, transform=train_transforms, cache_dir="/content/cache_train")
val_ds = PersistentDataset(data=val_files, transform=val_transforms, cache_dir="/content/cache_val")

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

print(f"Pipeline ready. Train cases: {len(train_files)}, Val cases: {len(val_files)}")

## UNet Model Training